In [ ]:
pip install vitmem

In [15]:
import torch
import torch.nn.functional as F
from torchvision import transforms
import numpy as np
from PIL import Image
import csv
import os
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from vitmem import ViTMem
from utils import sort_images
from utils import im_transforms_ratings as im_transforms

In [22]:
model_name = "vitmem"
if model_name == "vitmem":
    model = ViTMem()    
elif model_name == "memnet":
    from utils import MemNet
    model_name = "memnet"
    model = MemNet()
    checkpoint = torch.load("checkpoints/model.ckpt")
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

In [23]:
indices = sort_images(os.listdir("../images/n_back/"))

In [24]:
scores = []
for img_idx in range(len(indices)):
    image_path = f"../images/n_back/im{indices[img_idx]}.png"
    if model_name == "vitmem":
        memorability = model(image_path)
    elif model_name == "memnet":
        image = Image.open(image_path).convert("RGB")
        image = im_transforms(image)
        image = image.unsqueeze(0)
        memorability = model(image.clone()).item()
    scores.append(memorability)

In [25]:
images = [f'im{x}.png' for x in indices]

In [27]:
# Create dataframe
df = pd.DataFrame({
    "image_name": images,
    "image_index": indices,
    "score": np.array(scores)
})

# Save to CSV
df.to_csv("results/n_back.csv", index=False)